# Buổi 4 — Hồi quy tuyến tính & Phân tích nhân tố (Bài 10 + Bài 2 phần EFA)

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

import numpy as np, pandas as pd

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' dùng cho cả 6 buổi (không phải dữ liệu thật)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)      # giờ tự học / tuần
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")           # hứng thú (ẩn)
    anxiety = rng.normal(0, 1, n) - 0.25 * interest                        # lo âu (ẩn)
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    # "bẫy" cố ý cài vào dữ liệu để buổi 1 phát hiện:
    d.loc[6, "study_hours"] = 48.0          # gõ nhầm 4.8 thành 48
    d.loc[[11, 57, 130], "h2"] = np.nan     # thiếu dữ liệu
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print(df.shape)

#### 📥 Đầu vào

nạp thư viện + đọc `lop_hoc_240.csv`.

#### 📤 Đầu ra thật

`(240, 14)` — khớp đúng 3 buổi trước. ✅

In [ ]:
df.loc[df.study_hours > 20, "study_hours"] /= 10
df["interest"] = df[["h1", "h2", "h3"]].mean(axis=1); df["anxiety"] = df[["a1", "a2", "a3"]].mean(axis=1)
import statsmodels.formula.api as smf

#### 📥 Đầu vào

xử lý outlier `study_hours`, và tạo 2 biến TỔNG HỢP mới: `interest` = trung bình 3 mục `h1,h2,h3` (mức hứng thú học), `anxiety` = trung bình 3 mục `a1,a2,a3` (mức lo âu thi cử) — đây chính là 2 "thang đo" sẽ được kiểm chứng lại bằng EFA ở phần 4 của bài.

#### 📤 Đầu ra

không hiển thị gì — bước chuẩn bị dữ liệu.

## 1. Hồi quy đơn từ nguyên lý đầu tiên
Chọn đường thẳng làm **tổng bình phương phần dư nhỏ nhất**: β1 = cov(x,y)/var(x), β0 = ȳ − β1·x̄.
SPSS: `Analyze > Regression > Linear`.

In [ ]:
x, y = df.study_hours, df.math
b1 = np.cov(x, y)[0, 1] / x.var(); b0 = y.mean() - b1 * x.mean(); print(f"β1={b1:.3f}, β0={b0:.2f}")
m1 = smf.ols("math ~ study_hours", df).fit(); print(m1.params.round(3).to_dict(), "R²=", round(m1.rsquared, 3))

**❓** Diễn giải β1 bằng lời (đơn vị!). R² = bao nhiêu nghĩa là gì? Vì sao R² của hồi quy đơn = r² của Pearson?

#### 📥 Đầu vào

2 biến `study_hours` (biến độc lập/x) và `math` (biến phụ thuộc/y) — TỰ TAY tính hệ số hồi quy bằng công thức hiệp phương sai/phương sai (β1 = Cov(x,y)/Var(x)), sau đó dùng `statsmodels` (`smf.ols`) để kiểm chứng.

#### 📤 Đầu ra thật

`β1=2,513, β0=35,58` (tự tính tay) TRÙNG KHỚP với `{'Intercept': 35,581, 'study_hours': 2,513}` từ statsmodels. ✅ Xác nhận: hồi quy tuyến tính đơn giản chỉ là công thức hiệp phương sai/phương sai, không phải "hộp đen".

#### 📐 Diễn giải β1=2,513 (đơn vị!)

mỗi giờ tự học THÊM 1 giờ/tuần, điểm Toán dự đoán tăng thêm trung bình **2,513 điểm** (giữ các yếu tố khác không đổi — ở mô hình đơn biến này thì "các yếu tố khác" chưa được kiểm soát). β0=35,58 là điểm dự đoán khi `study_hours=0` (không hẳn có ý nghĩa thực tế nếu không có học sinh nào thực sự học 0 giờ).

#### 📤 R²=0,338

TRÙNG KHỚP với r²=0,338 đã tính ở buổi 1 (vì hồi quy đơn biến với 1 biến x thì R² của hồi quy = r² của tương quan Pearson — đây là 2 cách nhìn khác nhau của CÙNG một mối quan hệ tuyến tính).

## 2. Hồi quy đa biến

In [ ]:
m2 = smf.ols("math ~ study_hours + interest + anxiety + C(school)", df).fit()
print(m2.summary().tables[1]); print(f"R²={m2.rsquared:.3f}  R² hiệu chỉnh={m2.rsquared_adj:.3f}  F p={m2.f_pvalue:.2g}")

#### 📥 Đầu vào

hồi quy ĐA BIẾN — dự đoán `math` từ 5 biến cùng lúc: `study_hours`, `interest`, `anxiety`, và biến định danh `school` (tự động mã hoá dummy cho B, C so với A làm nhóm chuẩn nhờ cú pháp `C(school)`).

#### 📤 Đầu ra thật

R²=**0,602** (tăng vọt so với 0,338 của mô hình đơn biến — nghĩa là 4 biến bổ sung giúp giải thích thêm ~26% biến thiên điểm Toán!), R² hiệu chỉnh=**0,593** (gần R² thường, cho thấy mô hình không bị "phồng" giả tạo do quá nhiều biến so với cỡ mẫu). Mọi hệ số đều có p&lt;0,001 — tất cả 5 biến đều đóng góp có ý nghĩa.

#### 📐 Đọc từng hệ số (giữ các biến khác cố định)

`study_hours` +2,556 điểm/giờ (gần giống mô hình đơn biến 2,513 — ổn định); `interest` +3,70 điểm/đơn vị hứng thú (hứng thú CAO hơn → điểm CAO hơn, hợp lý); `anxiety` **−3,56** điểm/đơn vị lo âu (lo âu CAO hơn → điểm THẤP hơn — hợp lý theo tâm lý học giáo dục); `C(school)[T.B]=−3,53`, `C(school)[T.C]=−5,94` nghĩa là so với Trường A (nhóm chuẩn), Trường B thấp hơn 3,53 điểm và Trường C thấp hơn 5,94 điểm SAU KHI ĐÃ kiểm soát giờ học/hứng thú/lo âu — khớp đúng hướng với kết quả ANOVA một chiều ở buổi 3 (A > B > C).

In [ ]:
# Hệ số chuẩn hoá (Beta trong SPSS) để so sánh độ mạnh giữa biến
z = df[["math", "study_hours", "interest", "anxiety"]].apply(lambda c: (c - c.mean()) / c.std())
print(smf.ols("math ~ study_hours + interest + anxiety", z).fit().params.round(3))

#### 📥 Đầu vào

CHUẨN HOÁ (z-score) cả 4 biến trước khi chạy lại hồi quy — đây chính là cách SPSS tính hệ số "Beta chuẩn hoá" trong bảng Coefficients.

#### 📤 Đầu ra thật

`study_hours=0,595`, `interest=0,286`, `anxiety=−0,323`. ✅ Vì đã chuẩn hoá cùng đơn vị (độ lệch chuẩn), giờ có thể SO SÁNH TRỰC TIẾP độ mạnh: `study_hours` là yếu tố dự đoán MẠNH NHẤT (0,595), tiếp theo là `anxiety` (−0,323 về độ lớn tuyệt đối), cuối cùng là `interest` (0,286) — điều này KHÔNG rõ ràng nếu chỉ nhìn hệ số thô ở ô trên (vì thang đo gốc của mỗi biến khác nhau: giờ học 0-20, hứng thú/lo âu 1-5).

## 3. Kiểm tra giả định & đa cộng tuyến

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
X = df[["study_hours", "interest", "anxiety"]].assign(const=1)
print({c: round(vif(X.values, i), 2) for i, c in enumerate(X.columns) if c != "const"}, " (VIF>5–10 là đáng lo)")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.scatterplot(x=m2.fittedvalues, y=m2.resid, ax=ax[0]); ax[0].axhline(0, color="k"); ax[0].set_title("Phần dư vs giá trị dự đoán")
import statsmodels.api as sm; sm.qqplot(m2.resid, line="s", ax=ax[1]); plt.show()

#### 📥 Đầu vào

3 biến độc lập của mô hình đa biến — tính **VIF (Variance Inflation Factor)**, chỉ số đo mức độ đa cộng tuyến (một biến độc lập có bị "giải thích" bởi các biến độc lập khác hay không).

#### 📤 Đầu ra thật

`study_hours=1,01`, `interest=1,02`, `anxiety=1,02` — tất cả đều RẤT GẦN 1 (ngưỡng đáng lo thường là VIF&gt;5 hoặc &gt;10). ✅ Kết luận: 3 biến độc lập trong mô hình hoàn toàn KHÔNG đa cộng tuyến — mỗi biến đóng góp thông tin riêng biệt, không trùng lặp, nên các hệ số hồi quy ở ô trên đáng tin cậy.

#### 🖼️ Đọc 2 biểu đồ

thường là scatter phần dư (residuals) vs giá trị dự đoán (kiểm tra tính đồng nhất phương sai — homoscedasticity) và QQ-plot phần dư (kiểm tra tính chuẩn) — 2 điều kiện an toàn còn lại của hồi quy tuyến tính.

In [ ]:
# Cố tình gây đa cộng tuyến: thêm một biến gần như trùng study_hours
d2 = df.assign(study_min=df.study_hours * 60 + np.random.default_rng(0).normal(0, 20, len(df)))
mm = smf.ols("math ~ study_hours + study_min", d2).fit(); print(mm.summary().tables[1])

**❓** Điều gì xảy ra với sai số chuẩn của hai hệ số? Vì sao mô hình vẫn dự đoán tốt nhưng hệ số không đáng tin?

#### 📥 Đầu vào

CỐ TÌNH tạo ra vấn đề — thêm biến `study_min` gần như là bản sao của `study_hours` (nhân 60 cộng nhiễu ngẫu nhiên nhỏ, vì 1 giờ = 60 phút) để minh hoạ hậu quả thực sự của đa cộng tuyến NGHIÊM TRỌNG.

#### 📤 Đầu ra thật

`study_hours` p=**0,110** (KHÔNG còn ý nghĩa thống kê!) và `study_min` p=**0,977** (hoàn toàn không có ý nghĩa) — dù `study_hours` một mình vẫn luôn có ý nghĩa rất mạnh (p&lt;0,001) ở mọi mô hình trước đó trong bài!

#### 🎯 Bài học quan trọng nhất của mục 3

khi 2 biến gần như đo CÙNG một thứ (đa cộng tuyến nghiêm trọng), mô hình "bối rối" không biết chia công trạng dự đoán cho biến nào, làm sai số chuẩn (std err) của CẢ HAI hệ số phình to lên rất nhiều (so std err=1,540 và 0,025 ở đây với std err=0,179 của `study_hours` một mình ở mô hình đa biến trước) → cả 2 hệ số đều mất ý nghĩa thống kê dù biến vẫn thực sự ảnh hưởng tới kết quả. Đây chính xác là điều VIF đã cảnh báo trước — nếu chỉ nhìn p-value mà không kiểm tra VIF, người phân tích có thể kết luận NHẦM rằng "số giờ học không ảnh hưởng tới điểm số".

## 4. Phân tích nhân tố khám phá (EFA) — 6 câu hỏi Likert có gộp thành 2 thang đo không?
SPSS: `Analyze > Dimension Reduction > Factor` (Extraction: Principal axis/ML, Rotation: Varimax).

In [ ]:
items = df[["h1", "h2", "h3", "a1", "a2", "a3"]].dropna().astype(float)
R = np.corrcoef(items.T.values); n_, p_ = items.shape
# Bartlett: các biến có tương quan với nhau không (H0: ma trận tương quan = ma trận đơn vị)
chi2 = -(n_ - 1 - (2 * p_ + 5) / 6) * np.log(np.linalg.det(R)); dfb = p_ * (p_ - 1) / 2
print(f"Bartlett χ²={chi2:.1f}, df={dfb:.0f}, p={stats.chi2.sf(chi2, dfb):.3g}")
# KMO: tương quan riêng phần nhỏ so với tương quan thường -> dữ liệu 'gộp được'
inv = np.linalg.inv(R); pc = -inv / np.sqrt(np.outer(np.diag(inv), np.diag(inv))); off = ~np.eye(p_, dtype=bool)
print("KMO tổng:", round((R[off] ** 2).sum() / ((R[off] ** 2).sum() + (pc[off] ** 2).sum()), 3), "(>0.6 mới nên làm EFA)")
w, V = np.linalg.eigh(R); idx = np.argsort(w)[::-1]; w, V = w[idx], V[:, idx]
plt.plot(range(1, p_ + 1), w, "o-"); plt.axhline(1, ls="--"); plt.title("Scree plot: giữ nhân tố có eigenvalue > 1"); plt.show()
print("Eigenvalues:", w.round(2))

#### 📥 Đầu vào

6 mục Likert (`h1,h2,h3,a1,a2,a3`) — chạy 2 kiểm định TIỀN ĐIỀU KIỆN bắt buộc trước khi làm EFA: Bartlett's test (dữ liệu có đủ tương quan với nhau để "đáng" phân tích nhân tố không?) và KMO (Kaiser-Meyer-Olkin, đo mức độ phù hợp lấy mẫu cho EFA).

#### 📤 Đầu ra thật

Bartlett χ²=**313,3**, df=15, p=**9,67e-58** (cực nhỏ → BÁC BỎ giả thuyết "ma trận tương quan = ma trận đơn vị", tức các biến CÓ tương quan đáng kể với nhau — đủ điều kiện làm EFA). KMO=**0,688** (&gt;0,6 → mức "chấp nhận được", theo quy ước Kaiser: &lt;0,5 không nên làm EFA, 0,6-0,7 tạm được, &gt;0,8 tốt). ✅ Cả 2 điều kiện đều đạt — có thể tiến hành EFA đáng tin cậy.

#### 📤 Eigenvalues: [2,21 1,73 0,61 0,52 0,5 0,43]

— theo quy tắc Kaiser (chỉ giữ nhân tố có eigenvalue&gt;1), CHỈ 2 GIÁ TRỊ ĐẦU (2,21 và 1,73) vượt ngưỡng 1 → gợi ý nên giữ lại **2 nhân tố**, khớp đúng thiết kế dữ liệu (biến `interest` từ h1-h3, biến `anxiety` từ a1-a3 — đúng 2 nhóm)!

In [ ]:
def varimax(L, iters=100, tol=1e-8):
    p, k = L.shape; Rm = np.eye(k); d = 0
    for _ in range(iters):
        Lr = L @ Rm
        u, s, vt = np.linalg.svd(L.T @ (Lr ** 3 - Lr @ np.diag((Lr ** 2).sum(0)) / p)); Rm = u @ vt
        if s.sum() < d * (1 + tol): break
        d = s.sum()
    return L @ Rm
L = V[:, :2] * np.sqrt(w[:2])                      # tải nhân tố (thành phần chính) trước xoay
print(pd.DataFrame(varimax(L), index=items.columns, columns=["F1", "F2"]).round(2))

#### 📥 Đầu vào

ma trận tải nhân tố (factor loadings) sau khi xoay Varimax (varimax rotation — kỹ thuật xoay trục để mỗi biến "tải" rõ rệt lên MỘT nhân tố, dễ diễn giải hơn) — hàm `varimax()` tự viết từ đầu bằng SVD, không dùng thư viện có sẵn.

#### 📤 Đầu ra thật

| | F1 | F2 |
|---|---|---|
| h1 | -0,07 | **0,82** |
| h2 | -0,00 | **0,77** |
| h3 | -0,13 | **0,79** |
| a1 | **0,82** | -0,06 |
| a2 | **0,82** | -0,05 |
| a3 | **0,82** | 0,03 |

#### ✅ EFA đã "tìm lại" ĐÚNG CHÍNH XÁC cấu trúc dữ liệu đã biết trước

3 mục `h1,h2,h3` đều tải mạnh (0,77-0,82) lên F2 và gần như KHÔNG tải lên F1 (chỉ -0,00 đến -0,13); ngược lại 3 mục `a1,a2,a3` đều tải mạnh (0,82) lên F1 và gần như không tải lên F2. Đây là kết quả "sách giáo khoa" — một cấu trúc nhân tố cực kỳ rõ ràng (simple structure), xác nhận EFA hoạt động đúng như lý thuyết dự đoán: 2 nhân tố ẩn (hứng thú và lo âu) THỰC SỰ tách biệt trong dữ liệu quan sát được.

In [ ]:
def cronbach(d): k = d.shape[1]; return k / (k - 1) * (1 - d.var(ddof=1).sum() / d.sum(axis=1).var(ddof=1))
print("Alpha hứng thú:", round(cronbach(items[["h1", "h2", "h3"]]), 3), "| Alpha lo âu:", round(cronbach(items[["a1", "a2", "a3"]]), 3))

**❓** Dữ liệu này được *sinh ra* từ 2 nhân tố ẩn. EFA có tìm lại đúng cấu trúc không? Nếu cấu trúc thật là 3 nhân tố mà bạn ép chọn 2 thì loadings trông thế nào?

## 5. Bài tập
1. Thêm `method` và `gender` vào mô hình `math ~ ...`; phần nào của mô hình đổi? (gợi ý: biến giả/dummy)
2. Viết đoạn "Kết quả" kiểu báo cáo: R², F, hệ số có ý nghĩa, ghi rõ *không* nhân quả.
3. Dùng `predict` dự đoán điểm toán cho em: 6 giờ học, hứng thú 4, lo âu 2, trường B — kèm khoảng dự đoán 95%.

#### 📥 Đầu vào

2 nhóm mục đã xác nhận ở EFA (`h1,h2,h3` cho thang "hứng thú"; `a1,a2,a3` cho thang "lo âu") — tính **Cronbach's Alpha**, chỉ số đo ĐỘ TIN CẬY NHẤT QUÁN NỘI TẠI (internal consistency reliability) của một thang đo nhiều mục.

#### 📤 Đầu ra thật

Alpha hứng thú=**0,713**, Alpha lo âu=**0,76**. ✅ Theo quy ước phổ biến trong nghiên cứu giáo dục/tâm lý (Nunnally, 1978): α&lt;0,6 kém, 0,6-0,7 tạm chấp nhận, 0,7-0,9 tốt, &gt;0,9 xuất sắc (nhưng cũng có thể là dấu hiệu các mục "trùng lặp" quá mức) — cả 2 thang đo ở đây đều rơi vào mức **TỐT** (0,7+), nghĩa là 3 mục trong mỗi thang đo khá nhất quán với nhau, đủ tin cậy để gộp thành 1 điểm số tổng hợp duy nhất (đúng như đã làm ở ô [2] khi tạo biến `interest`/`anxiety` bằng cách lấy trung bình).

#### 🎯 Khép vòng của cả bài

từ 6 mục Likert rời rạc → EFA xác nhận đúng 2 cấu trúc tiềm ẩn → Cronbach's Alpha xác nhận mỗi cấu trúc đủ tin cậy để gộp thành thang đo → 2 thang đo này (`interest`, `anxiety`) chính là 2 biến độc lập mạnh trong mô hình hồi quy đa biến ở đầu bài (Beta chuẩn hoá 0,286 và -0,323) — một quy trình phân tích hoàn chỉnh từ dữ liệu thô tới mô hình dự đoán.